In [1]:
"""Clinical (% ED visits) vs. sequencing (rpkmf, 4-week rolling mean) time series,
one chart per virus: SARS-CoV-2, Flu (A+B combined), RSV (A+B combined).

This is the RAW (non-SV40-normalized) rolling rpkmf version.

Missing weeks (no sequencing sample that week, or -- for the clinical side --
any gap in the ED file) are bridged with a dashed segment rather than a solid
line or a blank gap.

"""


#Import required packages
import os
import matplotlib
matplotlib.use("Agg")
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from scipy.stats import pearsonr, spearmanr

#Set up input and output file formats
DATA_PATH = "Data_4a_Clinical_vs_Sequencing_Data_Final.xlsx"

COLOR_WW = "#3D2412"   # (brown) -- sequencing (rpkmf)
COLOR_ED = "#2a78d6"   # (blue) -- clinical (% ED visits)

GAP_LINESTYLE = (0, (4, 3))  # dashed, used by plot_gap_aware_series for missing-period bridges

VIRUSES = [
    {
        "sheet": "SARS-COV-2",
        "perc_col": "PERC_COVID",
        "rpkmf_col": "rpkmf_rolling_4wk",
        "title": "SARS-CoV-2",
        "out_path": "fig4a_sars_cov2_clinical_vs_sequencing_rolling_4wk.png",
    },
    {
        "sheet": "FLU",
        "perc_col": "PERC_INFLUENZA",
        "rpkmf_col": "flu_combined_rpkmf_rolling_4wk",
        "title": "Influenza (A+B combined)",
        "out_path": "fig4a_flu_clinical_vs_sequencing_rolling.png",
    },
   
    {
        "sheet": "RSV",
        "perc_col": "PERC_RSV",
        "rpkmf_col": "rsv_combined_rpkmf_rolling_4wk",
        "title": "RSV (A+B combined)",
        "out_path": "fig4a_rsv_clinical_vs_sequencing_rolling.png",
    },
]

def compute_correlations(sub, perc_col, rpkmf_col):
    """Pearson and Spearman correlation between the two series, restricted to
     weeks where both are present (pandas .corr() would otherwise silently
      pairwise-drop, but we want n reported explicitly).
    """
    paired = sub[[perc_col, rpkmf_col]].dropna()
    n = len(paired)
    if n < 2:
        return None, None,None, None, n
    pearson_r, pearson_p = pearsonr(paired[perc_col], paired[rpkmf_col])
    spearman_r, spearman_p = spearmanr(paired[perc_col], paired[rpkmf_col])
    return pearson_r, pearson_p, spearman_r, spearman_p, n


def plot_gap_aware_series(ax, x, y, color, label=None, linewidth=3.25, marker=None, markersize=None, zorder=3):
    """Plot a period-indexed series (x = one row per period on a regular
    weekly/monthly grid, already sorted; y may be NaN for periods with no
    data -- e.g. the result of merging onto the ED file's full weekly/monthly
    index) as solid segments between temporally-adjacent known points, and a
    dashed segment (GAP_LINESTYLE) bridging directly across any run of one or
    more missing periods. This makes a real gap in the underlying data read
    as "unknown" rather than a plain connected line (implies observed data
    across the gap) or a blank stretch (easy to misread as zero).

    x/y must be positional-index-aligned with the full period grid so that a
    gap in the data is exactly a gap in array position, not something inferred
    from timestamp arithmetic.
    """
    x = list(x)
    y = list(y)
    known = [i for i, v in enumerate(y) if pd.notna(v)]
    if label is not None:
        ax.plot([], [], color=color, linewidth=linewidth, marker=marker,
                 markersize=markersize, linestyle="-", label=label, zorder=zorder)
    for a, b in zip(known[:-1], known[1:]):
        style = "-" if b - a == 1 else GAP_LINESTYLE
        ax.plot([x[a], x[b]], [y[a], y[b]], color=color, linewidth=linewidth,
                 marker=marker, markersize=markersize, linestyle=style, zorder=zorder)


def _style_axes(ax_left, ax_right):
    ax_left.set_ylabel("Sequencing RPKMF \n (4-week rolling mean)", fontsize=14, color="black", fontweight="bold")
    ax_left.tick_params(axis="y", colors="black", labelsize=18)
    ax_left.spines["left"].set_color("black")

    ax_right.set_ylabel("% ED visits \n (Statewide)", fontsize=14, color="black", fontweight="bold")
    ax_right.tick_params(axis="y", colors="black", labelsize=18)
    ax_right.spines["right"].set_color("black")
    ax_right.spines["left"].set_visible(False)
    
    ax_left.spines["top"].set_visible(False)
    ax_right.spines["top"].set_visible(False)
    ax_left.spines["right"].set_visible(False)

    ax_left.set_ylim(bottom=0)
    ax_right.set_ylim(bottom=0)

    ax_left.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax_left.set_axisbelow(True)
    
def _format_p(p):
    return f"p = {p:.2e}"

def make_chart(virus, df):
    sub = df[["WEEK_ENDING", virus["perc_col"], virus["rpkmf_col"]]].sort_values("WEEK_ENDING").reset_index(drop=True)

    n_weeks = len(sub)
    n_with_rpkmf = sub[virus["rpkmf_col"]].notna().sum()
    print(f"{virus['title']}: {n_weeks} weeks total, {n_with_rpkmf} with sequencing data -> {virus['out_path']}")

    pearson_r, pearson_p, spearman_r, spearman_p, n_paired = compute_correlations(sub, virus["perc_col"], virus["rpkmf_col"])
    print(f"{virus['title']}: Pearson r={pearson_r:.3f} (p={pearson_p:.2e}), ", 
          f"Spearman rho={spearman_r:.3f} (p={spearman_p:.2e}) (n={n_paired} weeks)")
    
    fig, ax_left = plt.subplots(figsize=(16, 8))
    ax_right = ax_left.twinx()

    plot_gap_aware_series(ax_left, sub["WEEK_ENDING"], sub[virus["rpkmf_col"]], COLOR_WW,
                           label="Sequencing RPKMF (4-week rolling mean)")
    plot_gap_aware_series(ax_right, sub["WEEK_ENDING"], sub[virus["perc_col"]], COLOR_ED,
                           label="% ED visits (Statewide)")

    ax_left.set_yscale("symlog", linthresh=1)
    ax_left.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{y:.1f}"))
    _style_axes(ax_left, ax_right)
    ax_left.xaxis.set_major_locator(mdates.YearLocator())
    ax_left.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    ax_left.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax_left.set_xlabel("Date (Week ending)", fontsize=14, fontweight="bold")
    ax_left.tick_params(axis="both", labelsize=18)
    ax_left.text(
        0.985, 0.97,
        #f"Pearson r = {pearson_r:.2f} ({_format_p(pearson_p)})\n"
        f"Spearman ρ = {spearman_r:.2f} ({_format_p(spearman_p)})\n"
        f"n = {n_paired} weeks",
        transform=ax_left.transAxes, fontsize=18, va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.9),
        zorder=5)

    lines_left, labels_left = ax_left.get_legend_handles_labels()
    lines_right, labels_right = ax_right.get_legend_handles_labels()
    ax_left.legend(lines_left + lines_right, labels_left + labels_right,
                   loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=2, frameon=False, fontsize=18)
    ax_left.set_title(
        f"{virus['title']}: Wastewater Sequencing Signal vs. ED Visit Rate, Weekly Rolling Average",
        fontsize=18, fontweight="bold", loc="center", pad=20,
    )

    fig.tight_layout(rect=[0, 0.08, 1, 1])

    out_dir = os.path.dirname(virus["out_path"])
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    fig.savefig(virus["out_path"], dpi=300, bbox_inches="tight", pad_inches=0.3)
    fig.savefig(virus["out_path"].replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"Saved {virus['out_path']}")


def main():
    xl = pd.ExcelFile(DATA_PATH)
    for virus in VIRUSES:
        df = xl.parse(virus["sheet"])
        make_chart(virus, df)


if __name__ == "__main__":
    main()


SARS-CoV-2: 234 weeks total, 219 with sequencing data -> fig4a_sars_cov2_clinical_vs_sequencing_rolling_4wk.png
SARS-CoV-2: Pearson r=0.758 (p=4.46e-42),  Spearman rho=0.908 (p=5.35e-84) (n=219 weeks)
Saved fig4a_sars_cov2_clinical_vs_sequencing_rolling_4wk.png
Influenza (A+B combined): 234 weeks total, 213 with sequencing data -> fig4a_flu_clinical_vs_sequencing_rolling.png
Influenza (A+B combined): Pearson r=0.784 (p=1.46e-45),  Spearman rho=0.704 (p=3.23e-33) (n=213 weeks)
Saved fig4a_flu_clinical_vs_sequencing_rolling.png
RSV (A+B combined): 234 weeks total, 215 with sequencing data -> fig4a_rsv_clinical_vs_sequencing_rolling.png
RSV (A+B combined): Pearson r=0.444 (p=8.04e-12),  Spearman rho=0.672 (p=1.43e-29) (n=215 weeks)
Saved fig4a_rsv_clinical_vs_sequencing_rolling.png


In [1]:
"""Clinical (% ED visits) vs. sv40-normalized sequencing (rpkmf, 4-week rolling mean) time series,
one chart per virus: SARS-CoV-2, Flu (A+B combined), RSV (A+B combined).

This is the SV40-normalized rolling rpkmf version.

Missing weeks (no sequencing sample that week, or -- for the clinical side --
any gap in the ED file) are bridged with a dashed segment rather than a solid
line or a blank gap.
"""
#Import required packages
import os
import matplotlib
matplotlib.use("Agg")
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from scipy.stats import pearsonr, spearmanr

#Input data paths
DATA_PATH = "Data_4a_Clinical_vs_Sequencing_Data_Final.xlsx"

COLOR_WW = "#3D2412"   # dataviz categorical slot 1 (brown) -- sequencing (rpkmf)
COLOR_ED = "#2a78d6"   # dataviz categorical slot 2 (blue) -- clinical (% ED visits)

GAP_LINESTYLE = (0, (4, 3))  # dashed, used by plot_gap_aware_series for missing-period bridges

VIRUSES = [
    {
        "sheet": "SARS-COV-2",
        "perc_col": "PERC_COVID",
        "rpkmf_col": "normalized_rpkmf_rolling_4wk",
        "title": "SARS-CoV-2",
        "out_path": "sv40normalized_sars_cov2_clinical_vs_sequencing_rolling.png",
    },
    {
        "sheet": "FLU",
        "perc_col": "PERC_INFLUENZA",
        "rpkmf_col": "flu_combined_normalized_rpkmf_rolling_4wk",
        "title": "Influenza (A+B combined)",
        "out_path": "sv40normalized_flu_clinical_vs_sequencing_rolling.png",
    },
   
    {
        "sheet": "RSV",
        "perc_col": "PERC_RSV",
        "rpkmf_col": "rsv_combined_normalized_rpkmf_rolling_4wk",
        "title": "RSV (A+B combined)",
        "out_path": "sv40normalized_rsv_clinical_vs_sequencing_rolling.png",
    },
]

def compute_correlations(sub, perc_col, rpkmf_col):
    """Pearson and Spearman correlation between the two series, restricted to
     weeks where both are present (pandas .corr() would otherwise silently
      pairwise-drop, but we want n reported explicitly).
    """
    paired = sub[[perc_col, rpkmf_col]].dropna()
    n = len(paired)
    if n < 2:
        return None, None,None, None, n
    pearson_r, pearson_p = pearsonr(paired[perc_col], paired[rpkmf_col])
    spearman_r, spearman_p = spearmanr(paired[perc_col], paired[rpkmf_col])
    return pearson_r, pearson_p, spearman_r, spearman_p, n


def plot_gap_aware_series(ax, x, y, color, label=None, linewidth=3.25, marker=None, markersize=None, zorder=3):
    """Plot a period-indexed series (x = one row per period on a regular
    weekly/monthly grid, already sorted; y may be NaN for periods with no
    data -- e.g. the result of merging onto the ED file's full weekly/monthly
    index) as solid segments between temporally-adjacent known points, and a
    dashed segment (GAP_LINESTYLE) bridging directly across any run of one or
    more missing periods. This makes a real gap in the underlying data read
    as "unknown" rather than a plain connected line (implies observed data
    across the gap) or a blank stretch (easy to misread as zero).

    x/y must be positional-index-aligned with the full period grid so that a
    gap in the data is exactly a gap in array position, not something inferred
    from timestamp arithmetic.
    """
    x = list(x)
    y = list(y)
    known = [i for i, v in enumerate(y) if pd.notna(v)]
    if label is not None:
        ax.plot([], [], color=color, linewidth=linewidth, marker=marker,
                 markersize=markersize, linestyle="-", label=label, zorder=zorder)
    for a, b in zip(known[:-1], known[1:]):
        style = "-" if b - a == 1 else GAP_LINESTYLE
        ax.plot([x[a], x[b]], [y[a], y[b]], color=color, linewidth=linewidth,
                 marker=marker, markersize=markersize, linestyle=style, zorder=zorder)


def _style_axes(ax_left, ax_right):
    ax_left.set_ylabel("Sequencing RPKMF \n (4-week rolling mean)", fontsize=14, color="black", fontweight="bold")
    ax_left.tick_params(axis="y", colors="black",labelsize=18)
    ax_left.spines["left"].set_color("black")

    ax_right.set_ylabel("% ED visits \n (Statewide)", fontsize=14, color="black", fontweight="bold")
    ax_right.tick_params(axis="y", colors="black", labelsize=18)
    ax_right.spines["right"].set_color("black")
    ax_right.spines["left"].set_visible(False)
    
    ax_left.spines["top"].set_visible(False)
    ax_right.spines["top"].set_visible(False)
    ax_left.spines["right"].set_visible(False)

    ax_left.set_ylim(bottom=0)
    ax_right.set_ylim(bottom=0)

    ax_left.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax_left.set_axisbelow(True)
    
def _format_p(p):
    return f"p = {p:.2e}"

def make_chart(virus, df):
    sub = df[["WEEK_ENDING", virus["perc_col"], virus["rpkmf_col"]]].sort_values("WEEK_ENDING").reset_index(drop=True)

    n_weeks = len(sub)
    n_with_rpkmf = sub[virus["rpkmf_col"]].notna().sum()
    print(f"{virus['title']}: {n_weeks} weeks total, {n_with_rpkmf} with sequencing data -> {virus['out_path']}")

    pearson_r, pearson_p, spearman_r, spearman_p, n_paired = compute_correlations(sub, virus["perc_col"], virus["rpkmf_col"])
    print(f"{virus['title']}: Pearson r={pearson_r:.3f} (p={pearson_p:.2e}), ", 
          f"Spearman rho={spearman_r:.3f} (p={spearman_p:.2e}) (n={n_paired} weeks)")
    
    fig, ax_left = plt.subplots(figsize=(16, 8))
    ax_right = ax_left.twinx()

    plot_gap_aware_series(ax_left, sub["WEEK_ENDING"], sub[virus["rpkmf_col"]], COLOR_WW,
                           label="Sequencing RPKMF (4-week rolling mean)")
    plot_gap_aware_series(ax_right, sub["WEEK_ENDING"], sub[virus["perc_col"]], COLOR_ED,
                           label="% ED visits (Statewide)")

    ax_left.set_yscale("symlog", linthresh=1)
    ax_left.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{y:.1f}"))
    _style_axes(ax_left, ax_right)
    ax_left.xaxis.set_major_locator(mdates.YearLocator())
    ax_left.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
    ax_left.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax_left.set_xlabel("Date (Week ending)", fontsize=14, fontweight="bold")
    ax_left.tick_params(axis="both", labelsize=18)
    ax_left.text(
        0.985, 0.97,
        #f"Pearson r = {pearson_r:.2f} ({_format_p(pearson_p)})\n"
        f"Spearman ρ = {spearman_r:.2f} ({_format_p(spearman_p)})\n"
        f"n = {n_paired} weeks",
        transform=ax_left.transAxes, fontsize=18, va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.9),
        zorder=5)

    lines_left, labels_left = ax_left.get_legend_handles_labels()
    lines_right, labels_right = ax_right.get_legend_handles_labels()
    ax_left.legend(lines_left + lines_right, labels_left + labels_right,
                   loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=2, frameon=False, fontsize=18)
    ax_left.set_title(
        f"{virus['title']}: Wastewater Sequencing Signal vs. ED Visit Rate, Weekly Rolling Average",
        fontsize=18, fontweight="bold", loc="center", pad=20,
    )
    fig.tight_layout(rect=[0, 0.08, 1, 1])

    out_dir = os.path.dirname(virus["out_path"])
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    fig.savefig(virus["out_path"], dpi=300, bbox_inches="tight", pad_inches=0.3)
    fig.savefig(virus["out_path"].replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"Saved {virus['out_path']}")


def main():
    xl = pd.ExcelFile(DATA_PATH)
    for virus in VIRUSES:
        df = xl.parse(virus["sheet"])
        make_chart(virus, df)


if __name__ == "__main__":
    main()


SARS-CoV-2: 234 weeks total, 148 with sequencing data -> sv40normalized_sars_cov2_clinical_vs_sequencing_rolling.png
SARS-CoV-2: Pearson r=0.867 (p=4.23e-46),  Spearman rho=0.874 (p=1.10e-47) (n=148 weeks)
Saved sv40normalized_sars_cov2_clinical_vs_sequencing_rolling.png
Influenza (A+B combined): 234 weeks total, 148 with sequencing data -> sv40normalized_flu_clinical_vs_sequencing_rolling.png
Influenza (A+B combined): Pearson r=0.614 (p=9.90e-17),  Spearman rho=0.699 (p=5.14e-23) (n=148 weeks)
Saved sv40normalized_flu_clinical_vs_sequencing_rolling.png
RSV (A+B combined): 234 weeks total, 148 with sequencing data -> sv40normalized_rsv_clinical_vs_sequencing_rolling.png
RSV (A+B combined): Pearson r=0.548 (p=5.70e-13),  Spearman rho=0.723 (p=3.21e-25) (n=148 weeks)
Saved sv40normalized_rsv_clinical_vs_sequencing_rolling.png
